In [ ]:
# 03_violation_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# frequency-violation target
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                              recall_score, precision_recall_curve)

import features as f

TARGET = "violation_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()  # drop rows whose lead window is unresolvable

# --- Time-aware split ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): deliberately NOT using scale_pos_weight -- see
# features.py's scale_pos_weight() docstring for the full story.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}  (sanity check -- should NOT be 1)")

proba_val = model.predict_proba(X_val)[:, 1]
proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

# --- Threshold selection done on VAL, then frozen and applied to TEST (fixed
# 2026-07-14: previously the "best-F1" threshold was picked by scanning TEST's own PR
# curve for its best point -- reporting the best-in-hindsight operating point rather
# than what a threshold fixed in advance would actually achieve on unseen data. That's
# a mild but real optimistic bias in the headline precision/recall/confusion-matrix
# numbers. PR-AUC and F1@0.5 are unaffected -- neither involves picking a threshold
# from the set being evaluated.) ---
precision_val, recall_val, thresh_val = precision_recall_curve(y_val, proba_val)
f1s_val = 2 * precision_val * recall_val / (precision_val + recall_val + 1e-12)
best_idx_val = np.nanargmax(f1s_val[:-1])
best_thresh = thresh_val[best_idx_val]

preds_best = (proba_test >= best_thresh).astype(int)
test_precision = precision_score(y_test, preds_best, zero_division=0)
test_recall = recall_score(y_test, preds_best)
test_f1 = f1_score(y_test, preds_best)
print(f"Best-F1 threshold (selected on VAL): {best_thresh:.4f} -- applied to TEST: "
      f"F1={test_f1:.4f} (precision={test_precision:.4f}, recall={test_recall:.4f})")

idx95_val = np.where(precision_val[:-1] >= 0.95)[0]
if len(idx95_val):
    thresh_95 = thresh_val[idx95_val[-1]]  # laxest VAL threshold still hitting >=95% precision on VAL
    preds_95 = (proba_test >= thresh_95).astype(int)
    print(f"Threshold for >=95% precision (selected on VAL): {thresh_95:.4f} -- applied to TEST: "
          f"precision={precision_score(y_test, preds_95, zero_division=0):.4f}, "
          f"recall={recall_score(y_test, preds_95):.4f}")
else:
    print("No VAL threshold reaches >=95% precision")

# --- PR curve on TEST (the curve itself doesn't select a threshold, so no leakage
# concern here -- only marking a point on it does) ---
precision, recall, thresh = precision_recall_curve(y_test, proba_test)

plt.figure(figsize=(6, 6))
plt.plot(recall, precision, color="#e87ba4", linewidth=2)
plt.axhline(base_rate, color="#898781", linewidth=1, linestyle="--",
            label=f"random baseline ({base_rate:.3f})")
plt.scatter([test_recall], [test_precision], color="#0b0b0b", zorder=5,
            label="best-F1 point (threshold from VAL)", s=40)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve -- frequency-violation lead-time classifier")
plt.legend()
plt.tight_layout()
plt.show()

tp = int(((preds_best == 1) & (y_test == 1)).sum())
fp = int(((preds_best == 1) & (y_test == 0)).sum())
fn = int(((preds_best == 0) & (y_test == 1)).sum())
tn = int(((preds_best == 0) & (y_test == 0)).sum())
cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(cm, cmap="RdPu")
ax.set_xticks([0, 1], ["pred no-event", "pred event"])
ax.set_yticks([0, 1], ["actual no-event", "actual event"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title(f"Confusion matrix @ best-F1 threshold ({best_thresh:.3f}, from VAL)", fontsize=9)
plt.tight_layout()
plt.show()

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

plt.figure(figsize=(8, 6))
top_imp = importance.head(15)
norm = mpl.colors.Normalize(vmin=top_imp.min(), vmax=top_imp.max())
colors = mpl.colormaps["RdPu"](norm(top_imp.values))
plt.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors[::-1])
plt.xlabel("Split count")
plt.title("Feature importance -- frequency-violation lead-time classifier")
plt.tight_layout()
plt.show()

# --- Results (re-verified 2026-07-14, FIFTH pass -- methodology fix only, see the note
#     above the threshold-selection block: best-F1 threshold and the >=95%-precision
#     threshold are now chosen on VAL and frozen before being applied to TEST, instead
#     of being picked in hindsight on TEST's own curve.) ---
# PR-AUC: unchanged at 0.1567 (no feature/model change -- PR-AUC never involved
# threshold selection, so this number was never affected by the bug).
# Best-F1 threshold from VAL, applied to TEST: see the printed output for this run's
# exact precision/recall -- expect these to be close to, but not identical to, the old
# in-hindsight numbers (F1=0.2035, precision=17.7%, recall=24.0%), since VAL and TEST
# cover different months and the val-selected threshold is very unlikely to be exactly
# TEST's own optimum.
#
# Full history of this notebook's numbers, in order:
#  1. PR-AUC 0.0614 -- scale_pos_weight silently limiting training to 1 boosting round.
#  2. PR-AUC 0.0937 -- fixed scale_pos_weight + added solar_delta_mw/solar_roll8_std
#     (after the two-stage ramp->violation hypothesis was tested and found false).
#  3. PR-AUC 0.1186 -- removed share_res_pct + 11 corridor/cross-border columns (whole-
#     day aggregates broadcast to every slot; tested as a leakage concern, found to be
#     pure noise instead -- removing them helped, not hurt).
#  4. PR-AUC 0.1567 -- added freq_hz_delta and wind_delta_mw, the two next-highest
#     correlations with violation_lead from the same diagnostic scan that originally
#     found solar_delta_mw (freq_hz's own one-step delta: 0.0978, actually the single
#     highest correlation found in that whole scan; wind_delta_mw: 0.0361). Real,
#     verified gain: +32% PR-AUC over the previous version, and both new features have
#     genuine, nonzero importance in the trained model.
#  5. THIS version -- PR-AUC unchanged (0.1567); operating-point (best-F1, >=95%-
#     precision) numbers are now honestly out-of-sample instead of hindsight-optimal.


In [ ]:
# --- Appendix: shorter lead-window experiment (2026-07-14, FIFTH pass -- re-verified
#     with the val-selected-threshold fix from the main cell above) ---
# Does shrinking the lookahead window from 1-4 slots improve the violation classifier?
# features.py's add_violation_label() and build_feature_table() both take a lead_slots
# override for exactly this test.

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                              recall_score, precision_recall_curve)

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
resid = f.build_study1_residual_signal()  # computed once, reused across all window sizes


def run(lead_slots):
    feat = f.build_feature_table(scada, study1_residual=resid, violation_lead_slots=lead_slots)
    df = feat.dropna(subset=["violation_lead"]).copy()
    train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
    val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
    test = df[df["date"] >= "2026-01-01"]

    X_train, y_train = train[f.FEATURE_COLS], train["violation_lead"]
    X_val, y_val = val[f.FEATURE_COLS], val["violation_lead"]
    X_test, y_test = test[f.FEATURE_COLS], test["violation_lead"]

    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False)])

    proba_val = model.predict_proba(X_val)[:, 1]
    proba_test = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba_test)
    base_rate = y_test.mean()

    # threshold selected on VAL, frozen and applied to TEST -- same fix as the main cell
    precision_val, recall_val, thresh_val = precision_recall_curve(y_val, proba_val)
    f1s_val = 2 * precision_val * recall_val / (precision_val + recall_val + 1e-12)
    best_idx_val = np.nanargmax(f1s_val[:-1])
    best_thresh = thresh_val[best_idx_val]

    preds_best = (proba_test >= best_thresh).astype(int)
    test_precision = precision_score(y_test, preds_best, zero_division=0)
    test_recall = recall_score(y_test, preds_best)
    test_f1 = f1_score(y_test, preds_best)

    print(f"lead_slots={lead_slots}: best_iter={model.best_iteration_}  base_rate={base_rate:.4f}  "
          f"PR-AUC={pr_auc:.4f} (lift={pr_auc / base_rate:.2f}x)  best-F1={test_f1:.4f} "
          f"(P={test_precision:.4f} R={test_recall:.4f})")
    return {"lead_slots": lead_slots, "pr_auc": pr_auc, "lift": pr_auc / base_rate,
            "best_f1": test_f1}


results = []
for k in [4, 3, 2, 1]:
    results.append(run(k))

results_df = pd.DataFrame(results).sort_values("lead_slots", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 5))
norm = mpl.colors.Normalize(vmin=results_df["pr_auc"].min(), vmax=results_df["pr_auc"].max())
colors = mpl.colormaps["RdPu"](norm(results_df["pr_auc"].values))
ax.bar(results_df["lead_slots"].astype(str), results_df["pr_auc"], color=colors)
ax.set_xlabel("lead window (slots, 15 min each)")
ax.set_ylabel("PR-AUC")
ax.set_title("Violation classifier: PR-AUC vs lead-time window")
plt.tight_layout()
plt.show()

# --- Findings (re-verified 2026-07-14, FIFTH pass -- methodology fix only: best-F1 is
#     now selected on VAL and frozen before being applied to TEST, same fix as the main
#     cell. PR-AUC is unaffected by this fix (it never involved threshold selection),
#     so the PR-AUC/lift ranking across lead_slots should be unchanged from the fourth
#     pass; only the best-F1/precision/recall values may shift slightly.) ---
# See the printed output above for this run's exact numbers per lead_slots. Prior
# (fourth-pass, in-hindsight-threshold) values for reference:
# lead_slots=4 (shipped, 15-60 min): PR-AUC=0.1567 (5.13x lift)  best-F1=0.2035 (P=17.7% R=24.0%)
# lead_slots=3 (15-45 min):          PR-AUC=0.1612 (6.55x lift)  best-F1=0.2097 (P=21.0% R=21.0%)
# lead_slots=2 (15-30 min):          PR-AUC=0.2106 (11.62x lift) best-F1=0.2750 (P=22.7% R=34.9%)
# lead_slots=1 (15 min only):        PR-AUC=0.2185 (19.71x lift) best-F1=0.3212 (P=28.1% R=37.5%)
#
# The qualitative finding this appendix exists to support -- PR-AUC and lift improve
# monotonically as the lead window shrinks from 4 to 1 slot -- does not depend on the
# threshold-selection fix (PR-AUC is threshold-free), so it stands regardless of this
# methodology change.
#
# Still not changing the shipped default without an explicit product decision -- a
# 15-minute-only warning is a meaningfully different product than a 15-60-minute one,
# and (independent of this methodology fix) one more round of stability -- e.g. does
# this monotonic pattern hold under cross-validation, not just one time-aware split --
# would be worth having before treating it as settled.
